# 25.43 Laboratorio de Procesamiento de Señales — TP3 Beamforming y dirección de arribo

## Apuntar sin mover nada

Al comienzo del curso planteamos el problema de Starlink: un panel plano que atiende a miles
de usuarios sin partes móviles, apuntando **con sumas**. Este TP es ese problema, a tu escala:
un arreglo de sensores, una escena real, y los tres estimadores del curso — DAS, MVDR y MUSIC —
puestos a decir **de dónde viene** cada cosa.

Los laboratorios del módulo recorren la cadena sobre el arreglo sintético, y el taller del
17/11 la recorre sobre tu dataset. Lo que el TP agrega es llevarla a datos reales y hacerlo
**bien**: los parámetros elegidos con criterio, la incertidumbre medida, y la defensa escrita
de qué método usarías para tu escena.

---

## Qué se entrega

**Esta misma notebook, completada y corrida de arriba abajo**, más `dataset_doa.npz` y el script
que lo generó (`convertir.py`) junto a ella. La grabación original no hace falta: alcanza con
decir cuál es. No hay informe aparte: el análisis va en las celdas de markdown, al lado de la
figura que lo sostiene.

Antes de entregar, *Kernel → Restart & Run All*. Si no corre de punta a punta, no se corrige.

**Si usás inteligencia artificial** —ChatGPT, Claude, Copilot o cualquier otra—, anexá el
registro de **todas** las preguntas que le hiciste: completas, en orden y sin editar. Va en un
archivo aparte junto a la notebook (`registro_IA.md`).

## Cómo se evalúa

| Criterio | Peso |
|---|---|
| **Análisis crítico** | **40 %** |
| Corrección técnica | 30 % |
| Figuras | 15 % |
| Reproducibilidad | 15 % |

**El núcleo son las cinco partes.** El núcleo bien hecho es un 8; las extensiones suman por
encima.

> **Una advertencia sobre el método de trabajo.** Acá los parámetros son $\varepsilon$ (el
> loading), $p$ (el número de fuentes), la ventana de snapshots y la grilla de $\theta$. La
> manera equivocada es moverlos hasta que los picos queden lindos. La manera correcta es
> decidir con las reglas del curso —$K \gtrsim 2M$, los autovalores para $p$, el loading
> sobre el piso de ruido— y **decir qué regla usaste**. Se nota cuál de las dos hiciste.

## Las figuras

Ejes rotulados con unidades, y todo barrido angular en dB con su referencia dicha. Una figura
sin pie no argumenta: decora.

---

## Desde cuándo se puede hacer cada parte

| Parte | A partir del |
|---|---|
| 1. El arreglo y sus datos | **03/11** |
| 2. DAS: el barrido | **03/11** |
| 3. MVDR, y sus dos modos de falla | **10/11** |
| 4. MUSIC, y cuántas fuentes | **17/11** |
| 5. Los tres, comparados y defendidos | **17/11** |
| Extensiones | cada una dice la suya |

Es **a partir de**, no *recién en*. El taller del 17/11 se trabaja sobre esta notebook, con la
conversión hecha y las partes 1 a 3 ya corriendo: quedan cuatro días hasta la entrega, y son
para las partes 4 y 5 y la redacción — no para el arranque.

## Los datos

Un dataset **real** de arreglo de micrófonos, y la conversión a `dataset_doa.npz` con las
claves `X` (snapshots complejos $M \times K$, banda angosta), `d` (espaciado en $\lambda$) y
`descripcion`.

**Usá [LOCATA](https://zenodo.org/record/3630471), tarea 2, del conjunto de desarrollo**
(`dev.zip`, 6,2 GB; grabaciones 1 y 3): varios parlantes fijos, un arreglo fijo, 48 kHz y unos
0,5 s de reverberación. Citalo como pide [su página](https://locata.lms.tf.fau.de/datasets/). Son
archivos `.wav` y `.txt`, con etiquetas de actividad de voz para quedarte con los tramos en que
suena la fuente.

El arreglo DICIT de LOCATA contiene cuatro **ULA**:

| Espaciado | Micrófonos, en orden | $M$ | $d = \lambda/2$ en |
|---|---|---|---|
| 4 cm | 5, 6, 7, 9, 10 | 5 | 4,3 kHz |
| 8 cm | 4, 5, 7, 10, 11 | 5 | 2,1 kHz |
| 16 cm | 3, 4, 7, 11, 12 | 5 | 1,07 kHz |
| 32 cm | 1, 2, 3, 7, 12, 13, 14 | 7 | 536 Hz |

**Cuál usar, y a qué frecuencia, es parte del trabajo:** más apertura resuelve mejor, pero pide
frecuencias más bajas y aleja el campo lejano.

La verdad son las posiciones medidas. Con $\hat{\mathbf{e}}$ el versor
del último sensor de tu subarreglo al primero, y $\hat{\mathbf{u}}$ el versor del centro del
subarreglo a la fuente, $\sin\theta = \hat{\mathbf{u}} \cdot \hat{\mathbf{e}}$. **Estimá primero
y compará después.**

La celda de carga trae además el **generador sintético de cátedra** — el de los laboratorios.
No reemplaza al dataset real: sirve para **validar tu cadena** (con verdad conocida) antes de
soltarla sobre datos reales, y como red de seguridad si el dataset se resiste. Si terminás
entregando sobre el sintético, decilo y explicá qué se resistió: se corrige igual, con ese
contexto.

> **La banda angosta es tu responsabilidad.** Los datasets reales son de banda ancha: elegí
> una frecuencia de trabajo, y armá los snapshots complejos ahí. La STFT es la herramienta:
> cada ventana, en el bin de esa frecuencia, es un snapshot.
> Documentá $f$, el ancho, y el $d/\lambda$ resultante — el error más común del TP vive acá.

Un punto de partida para `convertir.py`. El largo y el salto los elegís y los defendés vos:

```python
import numpy as np
from scipy.signal import stft


def snapshots_stft(x, fs, f_trabajo, largo, salto):
    """x: M×N real, un sensor por fila, en el orden del subarreglo."""
    f, _, Z = stft(x, fs=fs, nperseg=largo, noverlap=largo - salto, axis=1)
    b = np.argmin(np.abs(f - f_trabajo))
    return Z[:, b, :], f[b]


# X, f = snapshots_stft(x, fs, f_trabajo, largo, salto)
# np.savez("dataset_doa.npz", X=X, d=d_metros * f / 343, descripcion="LOCATA, tarea 2, ...")
```


In [ ]:
# inline y no widget: la entrega tiene que renderizar sola para quien la corrija
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (11, 3.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

GRADOS = np.linspace(-90, 90, 1441)


def steering(theta_deg, M, d):
    return np.exp(-2j * np.pi * np.arange(M) * d * np.sin(np.deg2rad(theta_deg)))


print("Listo.")

In [ ]:
def cargar(ruta="dataset_doa.npz"):
    """Tu dataset, convertido: X (M×K complejo), d (en λ), descripcion."""
    datos = np.load(ruta)
    return datos["X"], float(datos["d"]), str(datos.get("descripcion", ruta))


def generar_ula(thetas, snrs_db, M=12, d=0.5, K=200, semilla=2543):
    """El generador sintético de cátedra: fuentes incorreladas + ruido blanco."""
    rng = np.random.default_rng(semilla)
    X = np.sqrt(0.5) * (rng.standard_normal((M, K)) + 1j * rng.standard_normal((M, K)))
    for theta, snr_db in zip(thetas, snrs_db):
        pot = 10 ** (snr_db / 10)
        s = np.sqrt(pot / 2) * (rng.standard_normal(K) + 1j * rng.standard_normal(K))
        X += np.outer(steering(theta, M, d), s)
    return X


# para validar la cadena: verdad conocida
X_val = generar_ula([-30, 10, 45], [8, 5, 10])

# el dataset del TP (generado por convertir.py):
X, D_LAMBDA, DESCRIPCION = cargar()
print(DESCRIPCION, X.shape)

In [ ]:
# Funciones compartidas por las partes 2 a 5, adaptadas de las clases 14 a 16
from scipy.signal import find_peaks


def covarianza(X):
    """R estimada a partir de los K snapshots (columnas de X)."""
    return X @ X.conj().T / X.shape[1]


def posiciones_ula(M, d, sensor=None, corrimiento=0.0, escala=1.0):
    """Posiciones de los sensores en λ según el modelo. Por defecto, el ULA ideal.
    sensor/corrimiento: corre un sensor esa fracción de λ; escala: error relativo en d."""
    pos = np.arange(M) * d * escala
    if sensor is not None:
        pos[sensor] += corrimiento
    return pos


def matriz_steering(M, d, grados=GRADOS, posiciones=None):
    """Un vector de dirección por columna, uno por cada θ de la grilla.
    Con posiciones=None es el steering del setup; si no, el de ese modelo del arreglo."""
    if posiciones is None:
        return np.array([steering(th, M, d) for th in grados]).T
    return np.exp(-2j * np.pi * np.outer(posiciones, np.sin(np.deg2rad(grados))))


def das(R, d, grados=GRADOS, posiciones=None):
    """Periodograma espacial: a^H R a / M², para cada θ."""
    M = R.shape[0]
    A = matriz_steering(M, d, grados, posiciones)
    return np.real(np.sum(A.conj() * (R @ A), axis=0)) / M**2


def piso_ruido(R, p):
    """Media de los M − p autovalores más chicos de R."""
    autovalores = np.sort(np.linalg.eigvalsh(R))
    return np.mean(autovalores[: R.shape[0] - p])


def mvdr(R, d, eps=0.0, grados=GRADOS, posiciones=None):
    """Capon: 1 / (a^H (R + εI)⁻¹ a), para cada θ."""
    M = R.shape[0]
    A = matriz_steering(M, d, grados, posiciones)
    Ri = np.linalg.inv(R + eps * np.eye(M))
    return 1 / np.real(np.sum(A.conj() * (Ri @ A), axis=0))


def music(R, d, p, grados=GRADOS, posiciones=None):
    """Pseudoespectro: 1 / ||E_n^H a||², con E_n los M − p autovectores de ruido."""
    M = R.shape[0]
    _, V = np.linalg.eigh(R)                 # autovalores de menor a mayor
    ruido = V[:, : M - p]
    A = matriz_steering(M, d, grados, posiciones)
    return 1 / np.linalg.norm(ruido.conj().T @ A, axis=0) ** 2


def a_db(P):
    """Barrido en dB respecto de su propio máximo."""
    return 10 * np.log10(P / P.max())


def picos(P, n, grados=GRADOS):
    """Los θ de los n picos más altos del barrido, ordenados."""
    idx, _ = find_peaks(P)
    mejores = idx[np.argsort(P[idx])[::-1][:n]]
    return np.sort(grados[mejores])


def potencia_db(P, theta, grados=GRADOS):
    """Valor del barrido en la dirección θ, en dB absolutos (sin normalizar)."""
    return 10 * np.log10(P[np.argmin(np.abs(grados - theta))])


def doa_por_bloques(X, estimador, n_fuentes, n_bloques, tolerancia):
    """Parte X en bloques contiguos de snapshots y estima las DOA en cada uno.
    estimador(R) devuelve un barrido. Cada pico se asocia a la DOA de referencia
    (la de todos los snapshots) más cercana; si ninguno cae a menos de `tolerancia`
    grados, esa fuente queda NaN en ese bloque."""
    referencia = picos(estimador(covarianza(X)), n_fuentes)
    doas = np.full((n_bloques, n_fuentes), np.nan)
    for b, Xb in enumerate(np.array_split(X, n_bloques, axis=1)):
        encontrados = picos(estimador(covarianza(Xb)), n_fuentes)
        for j, ref in enumerate(referencia):
            if encontrados.size:
                cerca = encontrados[np.argmin(np.abs(encontrados - ref))]
                if abs(cerca - ref) < tolerancia:
                    doas[b, j] = cerca
    return referencia, doas

---

# Parte 1 — El arreglo y sus datos

> **A partir del 03/11**

Antes de ningún estimador: **qué mediste**.

**Qué hay que hacer.**

1. Documentá el arreglo: qué subarreglo elegiste y por qué, $M$, espaciado físico, y el
   $d/\lambda$ a tu frecuencia de trabajo. Si $d > \lambda/2$, decilo ya: tus barridos van a
   tener réplicas, y conviene saber dónde antes de interpretarlas. `steering` mide $\theta$
   desde la normal al arreglo, con $\theta > 0$ del lado del primer sensor.
2. Armá los snapshots de banda angosta en `convertir.py`, y justificá acá la ventana: cuántos
   son, cuán separados,
   y si la escena se puede suponer **quieta** durante ese tramo — es la estacionariedad de
   este módulo.
3. Formá $\hat{\mathbf{R}}$ y mirala: diagonal (¿algún sensor muerto o desbalanceado?),
   traza, y el espectro de autovalores. Guardá ese gráfico: la parte 4 lo retoma.
4. **Validá la cadena sobre el sintético** (`X_val`, verdad en el código) antes de soltarla
   sobre el dataset: si tu barrido no clava −30°, 10° y 45° ahí, no va a clavar nada en el
   dato real.

**Qué hay que responder.**

- ¿Cuánto vale $K/M$?
- ¿Qué apertura tenés en $\lambda$, y qué resolución de Rayleigh implica?
- ¿Se sostienen las dos hipótesis? Banda angosta, y campo lejano: compará la distancia a cada
  parlante con tu apertura.

In [ ]:
# Parte 1 — punto 4: Validación de la cadena sobre el sintético: la verdad es −30°, 10° y 45°
VERDAD_VAL = [-30, 10, 45]
M_val, d_val, p_val = X_val.shape[0], 0.5, 3

R_val = covarianza(X_val)
eps_val = 10 * piso_ruido(R_val, p_val)      # 10 dB sobre el piso de ruido
barridos_val = {
    "DAS": das(R_val, d_val),
    "MVDR": mvdr(R_val, d_val, eps=eps_val),
    "MUSIC": music(R_val, d_val, p_val),
}

for nombre, P in barridos_val.items():
    print(f"{nombre:6s} picos en {picos(P, p_val)} °")
print(f"piso de ruido {piso_ruido(R_val, p_val):.2f}, ε = {eps_val:.2f}")

fig, ax = plt.subplots()
for nombre, P in barridos_val.items():
    ax.plot(GRADOS, a_db(P), lw=1.4, label=nombre)
for th in VERDAD_VAL:
    ax.axvline(th, color="k", ls=":", lw=0.8)
ax.set_xlim(-90, 90)
ax.set_ylim(-40, 1)
ax.set_xlabel("θ [grados]")
ax.set_ylabel("dB (cada barrido a su máximo)")
ax.legend()
plt.tight_layout()
plt.show()

*Tu análisis de la parte 1, acá.*

---

# Parte 2 — DAS: el barrido

> **A partir del 03/11**

El periodograma espacial, sobre tu escena.

**Qué hay que hacer.**

1. El barrido DAS completo, en dB, con la grilla de $\theta$ declarada.
2. Marcá los picos que interpretás como fuentes, y el ancho de cada uno contra el Rayleigh de
   tu apertura: ¿son fuentes, o lóbulos laterales de otra?
3. Si tu $d/\lambda$ lo permite, mostrá (o descartá con la cuenta) las réplicas de grating.

**Qué hay que responder.**

- ¿Cuántas fuentes *afirmarías* con DAS solo, y con qué duda?
- ¿Qué limita acá: la apertura, el ruido, o los lóbulos laterales?

In [ ]:
# Parte 2 — tu código acá


*Tu análisis de la parte 2, acá.*

---

# Parte 3 — MVDR, y sus dos modos de falla

> **A partir del 10/11**

El barrido de Capon sobre tu $\hat{\mathbf{R}}$ — con sus dos modos de falla vigilados,
porque acá la $\mathbf{R}$ es estimada y el steering es el de *tu* modelo del arreglo.

**Qué hay que hacer.**

1. El barrido MVDR con diagonal loading. Elegí $\varepsilon$ como en el laboratorio de MVDR,
   10 dB sobre el piso de ruido, que leés en los autovalores chicos de tu $\hat{\mathbf{R}}$.
   Mostrá el barrido con y sin loading.
2. **Snapshots:** partí tu registro y mostrá cómo cambia el barrido con $K/2$ y con $K$.
   ¿Estás del lado bueno de la regla $2M$?
3. **Desajuste:** corré en tu modelo del arreglo la posición de un sensor, una fracción de
   $\lambda$. En la dirección de cada fuente, compará en dB absolutos la potencia
   que dan DAS y MVDR con el modelo correcto y con el perturbado. Es el
   contraejemplo de la clase, sobre tu dato: el desajuste que importa es el de *tu*
   calibración, no el del sintético.

**Qué hay que responder.**

- ¿Qué ganó MVDR sobre DAS en tu escena, y dónde perdió?
- ¿Tu escena tiene interferentes fuertes que justifiquen los nulos, o MVDR está de adorno?
- ¿Qué le pasa al barrido de MVDR si el error es del 2 % en $d$, y por qué?

In [ ]:
# Desajuste del modelo del arreglo sobre el sintético (base de la Parte 3.3)
SENSOR, CORRIMIENTO = 1, 0.1                         # corre el sensor 1 un décimo de λ

modelos = {
    "correcto": posiciones_ula(M_val, d_val),
    f"sensor {SENSOR} +{CORRIMIENTO}λ": posiciones_ula(M_val, d_val, sensor=SENSOR, corrimiento=CORRIMIENTO),
    "d con +2 %": posiciones_ula(M_val, d_val, escala=1.02),
}
metodos = {
    "DAS": lambda pos: das(R_val, d_val, posiciones=pos),
    "MVDR sin carga": lambda pos: mvdr(R_val, d_val, posiciones=pos),
    "MVDR con carga": lambda pos: mvdr(R_val, d_val, eps=eps_val, posiciones=pos),
}
barridos = {(m, mod): fn(pos) for m, fn in metodos.items() for mod, pos in modelos.items()}

# potencia en la dirección de cada fuente, en dB absolutos
for th in VERDAD_VAL:
    print(f"fuente en {th}°")
    for m in metodos:
        valores = [potencia_db(barridos[(m, mod)], th) for mod in modelos]
        perdidas = "   ".join(f"{mod}: {v:5.1f} dB ({v - valores[0]:+.1f})"
                               for mod, v in zip(modelos, valores))
        print(f"   {m:15s} {perdidas}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
for ax, m in zip(axes, metodos):
    for mod in modelos:
        ax.plot(GRADOS, 10 * np.log10(barridos[(m, mod)]), lw=1.2, label=mod)
    for th in VERDAD_VAL:
        ax.axvline(th, color="k", ls=":", lw=0.8)
    ax.set_title(m)
    ax.set_xlim(-90, 90)
    ax.set_xlabel("θ [grados]")
axes[0].set_ylabel("potencia [dB absolutos]")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

*Tu análisis de la parte 3, acá.*

---

# Parte 4 — MUSIC, y cuántas fuentes

> **A partir del 17/11**

La pregunta que MUSIC obliga a contestar antes de contestar la tuya.

**Qué hay que hacer.**

1. El espectro de autovalores de la parte 1, ahora leído: elegí $p$ y defendé la elección.
2. El pseudoespectro MUSIC con tu $p$ — y con $p - 1$ y $p + 1$ al lado: ¿cuán sensible es lo
   que afirmás?
3. **DOA con incertidumbre:** partí tus snapshots en bloques, estimá las DOA en cada bloque, y
   reportá media y dispersión por fuente. Un $\theta$ sin incertidumbre no es una medición.
   Compará recién entonces con el $\theta$ que sale de las posiciones medidas.
4. El control de sanidad del ULA: el arreglo no distingue adelante de atrás, y lo que mide es
   $\sin\theta$ respecto de su eje: una fuente a otra altura cae en el mismo cono. Decí qué
   ambigüedad tiene tu geometría y cómo la tratás al reportar.

**Qué hay que responder.**

- ¿El pseudoespectro te dio picos donde DAS no veía nada? ¿Les creés? ¿Por qué?
- Si tus fuentes fueran coherentes (multipath), ¿qué le pasaría a esta parte? Una oración —
  la extensión B lo desarrolla.

In [ ]:
# DOA por bloques sobre el sintético: media y dispersión por fuente (base de la Parte 4.3)
N_BLOQUES = 5
K_val = X_val.shape[1]
tol_val = np.rad2deg(1 / (M_val * d_val)) / 2       # medio Rayleigh

ref_val, doas_val = doa_por_bloques(X_val, lambda R: music(R, d_val, p_val),
                                     n_fuentes=p_val, n_bloques=N_BLOQUES, tolerancia=tol_val)

print(f"{N_BLOQUES} bloques de {K_val // N_BLOQUES} snapshots (2M = {2 * M_val}), "
      f"tolerancia ±{tol_val:.1f}°")
for j, verdad in enumerate(VERDAD_VAL):
    col = doas_val[:, j]
    print(f"verdad {verdad:6.1f}°   media {np.nanmean(col):7.2f}°   desvío {np.nanstd(col, ddof=1):5.2f}°"
          f"   bloques sin pico: {np.sum(np.isnan(col))}")

*Tu análisis de la parte 4, acá.*

---

# Parte 5 — Los tres, comparados y defendidos

> **A partir del 17/11**

Esta parte tiene poco código, y es la que más pesa.

**Qué hay que hacer.**

1. Los tres barridos en los mismos ejes, cada uno a su máximo, con las DOA de la parte 4
   marcadas.
2. La tabla de tu escena: para cada método, qué estimó, qué supuso, y dónde se notó ese
   supuesto **en tus datos**.
3. **Resolución y ruido, con el generador**, donde conocés la verdad: dos fuentes de igual
   potencia, con tu $M$ y tu $K$. Barré la separación, para varias SNR, y mostrá desde qué
   separación cada método da dos picos. Usá varias semillas y decí cuántas.

**Qué hay que responder.**

- Para *tu* escena y *tu* pregunta: ¿cuál de los tres usarías, y **qué estás suponiendo** al
  elegirlo? Es la pregunta del cierre de estimación espectral, ahora cerrando el curso.
- En el punto 3, ¿cuál método depende más de la SNR? ¿Te sorprende, viendo qué supone cada uno?
- ¿Qué medirías distinto si pudieras repetir la adquisición?

> Las elecciones distintas pueden estar bien las dos. Lo que se corrige es el argumento,
> apoyado en las figuras de las partes 2 a 4.

In [ ]:
# Parte 5.3 — resolución contra separación y SNR, con el generador de cátedra
import time

M_RES, K_RES = X.shape                # el M y el K de la escena del TP
D_RES = D_LAMBDA
SEPARACIONES = np.arange(1, 41, 1.0)  # grados entre las dos fuentes
SNRS_RES = [0, 10, 20]                # dB por fuente, contra ruido de potencia 1
SEMILLAS = 50
CENTRO = 0.0                          # las fuentes en CENTRO ± Δ/2: el caso más favorable (broadside)

pos_res = posiciones_ula(M_RES, D_RES)          # con posiciones explícitas, el steering sale vectorizado
estimadores_res = {
    "DAS": lambda R: das(R, D_RES, posiciones=pos_res),
    "MVDR": lambda R: mvdr(R, D_RES, eps=10 * piso_ruido(R, 2), posiciones=pos_res),
    "MUSIC": lambda R: music(R, D_RES, 2, posiciones=pos_res),
}


def resuelve(P, verdad, sep):
    """Dos picos, y cada uno a menos de media separación de su fuente."""
    est = picos(P, 2)
    return est.size == 2 and np.all(np.abs(est - np.sort(verdad)) < sep / 2)


t0 = time.perf_counter()
prob = {m: np.zeros((len(SNRS_RES), len(SEPARACIONES))) for m in estimadores_res}
for i, snr in enumerate(SNRS_RES):
    for j, sep in enumerate(SEPARACIONES):
        verdad = [CENTRO - sep / 2, CENTRO + sep / 2]
        for semilla in range(SEMILLAS):
            R = covarianza(generar_ula(verdad, [snr, snr], M=M_RES, d=D_RES, K=K_RES, semilla=semilla))
            for m, est in estimadores_res.items():
                prob[m][i, j] += resuelve(est(R), verdad, sep) / SEMILLAS
print(f"{len(SNRS_RES)} SNR × {len(SEPARACIONES)} separaciones × {SEMILLAS} semillas "
      f"en {time.perf_counter() - t0:.0f} s")

# separación mínima a la que cada método resuelve en al menos la mitad de las semillas
rayleigh = np.rad2deg(1 / (M_RES * D_RES))
print(f"M = {M_RES}, K = {K_RES}, d = {D_RES:.3f}λ — Rayleigh ≈ {rayleigh:.1f}°")
print(f"{'SNR':>6s}" + "".join(f"{m:>10s}" for m in estimadores_res))
for i, snr in enumerate(SNRS_RES):
    fila = []
    for m in estimadores_res:
        ok = np.nonzero(prob[m][i] >= 0.5)[0]
        fila.append(f"{SEPARACIONES[ok[0]]:9.0f}°" if ok.size else "      > 40°")
    print(f"{snr:4d} dB" + "".join(fila))

fig, axes = plt.subplots(1, len(SNRS_RES), figsize=(11, 3.6), sharey=True)
for ax, i in zip(axes, range(len(SNRS_RES))):
    for m in estimadores_res:
        ax.plot(SEPARACIONES, prob[m][i], lw=1.4, label=m)
    ax.axvline(rayleigh, color="k", ls=":", lw=0.8, label="Rayleigh")
    ax.axhline(0.5, color="0.6", lw=0.8)
    ax.set_title(f"SNR {SNRS_RES[i]} dB por fuente")
    ax.set_xlabel("separación [grados]")
axes[0].set_ylabel(f"fracción de las {SEMILLAS} semillas que resuelven")
axes[-1].legend(fontsize=8)
plt.tight_layout()
plt.show()

*Tu defensa, acá. Es la sección más importante de la entrega.*

---

# Extensiones

Suman por encima del núcleo. **Una hecha bien vale más que tres empezadas.**

**A. ESPRIT, sin barrer** · *a partir del 17/11*
Implementá ESPRIT sobre los dos subarreglos de tu ULA y compará sus DOA con las de MUSIC:
precisión, tiempo de cómputo, y qué pasa con la incertidumbre de la parte 4. ¿Cuándo conviene
el barrido de MUSIC, y cuándo no?

**B. El eco que rompe el subespacio** · *a partir del 17/11*
Modificá el generador para obtener dos fuentes **coherentes** (la misma
señal, retardada y atenuada — un multipath). Mostrá que MUSIC se rompe, aplicá spatial
smoothing, y mostrá la reparación y la apertura que se pierde. Después decí si tu dataset real puede tener este problema.

**C. El arreglo casero** · *libre*
Cuatro micrófonos (o cuatro celulares grabando a la vez), una regla, y palmas desde ángulos
medidos con cinta en el piso. Sincronizá por el pico, armá los snapshots, y localizá. El
protocolo y sus miserias —sincronía, $d/\lambda$, reverberación— valen tanto como el
resultado.

**D. La fuente que se mueve** · *a partir del 17/11*
Si tu dataset tiene una fuente móvil (o generala): DOA por ventanas deslizantes y la
trayectoria $\theta(t)$. ¿Qué ventana elegís, y qué compromiso estás jugando? — es el de la STFT,
en el plano ángulo-tiempo.

---

## Antes de entregar

- *Kernel → Restart & Run All*, y que corra entero — con `dataset_doa.npz` junto a la notebook.
- `convertir.py` junto a la notebook, con los parámetros que usaste.
- Todas las figuras con ejes rotulados, unidades y pie.
- Las celdas de análisis contestadas, no vacías. **Son el 40 %.**
- Nombres de los integrantes en la celda de abajo.

**Los seis errores que más aparecen.**

1. **El $d/\lambda$ equivocado.** El espaciado físico es fijo; $\lambda$ depende de la
   frecuencia de trabajo que elegiste. Documentala, y rehacé la cuenta si cambiás de banda.
2. Snapshots de banda ancha metidos en un modelo de banda angosta: los picos se corren y
   nadie sabe por qué. La conversión de la parte 1 es la mitad del TP.
3. Grilla de $\theta$ de 1° y DOA reportada con dos decimales: tu precisión no puede ser
   mejor que tu grilla ni que tu dispersión entre bloques.
4. $p$ elegido para que MUSIC muestre lo que esperabas, sin mirar los autovalores.
5. Comparar métodos con ventanas de snapshots distintas y atribuir la diferencia al método.
6. Reportar DOA sin incertidumbre, o sin decir la ambigüedad adelante-atrás del ULA.

---

**Grupo:**

**Integrantes:**

**Extensiones hechas:**